# **Laboratorio 9 - Visión por Computadora**

- Paula Barillas - 22764
- Gerardo Pineda - 22880
- Mónica Salvatierra - 22249

Link del repositorio: https://github.com/paulabaal12/LAB9-VCP

## **Task 1**

**Además de las cámaras de seguridad con YOLO, los organizadores del Akihabara Fest quieren instalar "Cabinas Fotográficas de Realidad Aumentada (AR)" para los asistentes VIP. El objetivo de estas cabinas es doble:**

1. Eliminar automáticamente el fondo real de la convención y reemplazarlo por escenarios de anime (ej. La Aldea Oculta de la Hoja de Naruto o Neo-Tokyo de Akira), simulando un "Chroma Key" (pantalla verde) perfecto sin usar tela verde.

2. Identificar específicamente la espada o báculo mágico que sostiene el usuario para superponerle un 
efecto de "brillo de energía" digital, diferenciándolo de las armas de otros amigos que estén en la 
misma foto. 
La mesa directiva no sabe qué modelo de IA usar y le pide a usted investigar dos arquitecturas famosas: U
Net y Mask R-CNN. 
Entonces, realice una investigación en artículos o documentación técnica oficial y redacte un reporte 
ejecutivo (máximo 2 página) respondiendo con rigor técnico a lo siguiente: 
1. Diferencia Fundamental: Defina con sus propias palabras la diferencia exacta a nivel de píxel entre 
la Segmentación Semántica y la Segmentación de Instancia. 
2. El Caso U-Net: Explique por qué la arquitectura U-Net (Segmentación Semántica) es ideal y altamente 
eficiente para la tarea 1 (separar a todos los humanos del fondo), pero explique técnicamente por 
qué fracasaría si intentamos usarla para la tarea 2 si hay dos réplicas de espadas idénticas cruzadas 
en la foto. 
3. El Caso Mask R-CNN: Explique cómo la arquitectura Mask R-CNN (Segmentación de Instancia) 
resuelve el problema anterior basándose en su naturaleza de "dos etapas" (recordando que hereda 
de Faster R-CNN). ¿Por qué Mask R-CNN sí podría iluminar una espada de rojo y otra de azul, incluso 
si se están tocando en la imagen?

**Informe:**

### **1. Diferencia Fundamental: Segmentación Semántica vs. Segmentación de Instancia**

A nivel de píxel, ambas técnicas colorean cada píxel de la imagen con una etiqueta, pero con una diferencia en qué significa esa etiqueta.

Segmentación Semántica asigna a cada píxel una clase. Todos los píxeles que pertenecen a objetos de la misma clase reciben el mismo color/etiqueta, sin importar cuántos objetos de ese tipo existan. Si hay tres personas en la imagen, los píxeles de las tres personas se colorean con el mismo valor. 

Segmentación de Instancia va un paso más allá: asigna a cada píxel una clase y un identificador de instancia único. Si hay tres personas, cada una tiene su propio ID independiente.

### **2. El Caso U-Net: Fortaleza en Tarea 1, Fracaso en Tarea 2**

U-Net, es ideal para la semana 1, para eliminar el fondo. Es una red completamente convolucional con arquitectura encoder-decoder simétrica conectada por skip connections. Su encoder contrae la imagen capturando contexto semántico global, mientras el decoder la reconstruye a resolución original recuperando detalles espaciales finos.

Hace píxel a píxel sobre toda la imagen en un solo forward pass. No necesita distinguir entre individuos, solo distinguir clase humano vs. clase fondo. Las skip connections preservan bordes y contornos finos (cabello, dedos), produciendo un chroma key nítido. Es computacionalmente ligera, apta para tiempo casi real en cabinas fotográficas.


Si se usara la semana 2, el fracaso es arquitectural, no de entrenamiento. U-Net produce un único mapa de segmentación donde cada clase tiene un canal de salida. Si dos espadas idénticas se cruzan en la imagen, sus píxeles son visualmente indistinguibles para la red, lo que es la misma textura, mismo color, misma clase. U-Net etiquetará ambas como "arma" (un solo canal activado), pero es estructuralmente incapaz de generar dos máscaras separadas para dos instancias de la misma clase. No existe en su salida el concepto de "espada número 1" y "espada número 2". Intentar iluminar una de rojo y otra de azul es imposible porque la red nunca produjo dos objetos distintos, solo una región de clase "arma" continua y fusionada. Donde las espadas se tocan, los píxeles se mezclan en una sola región sin fronteras internas.

### **3. El Caso Mask R-CNN: Resolución por Segmentación de Instancia en Dos Etapas**

Mask R-CNN hereda la arquitectura de Faster R-CNN y le añade una tercera rama paralela de segmentación. Se divide en dos etapas:

**Etapa 1 — Region Proposal Network (RPN):** Heredada de Faster R-CNN, esta sub-red escanea la imagen con anclas y propone Regiones de Interés, cajas delimitadoras candidatas que probablemente contienen objetos. Crucialmente, cada propuesta es independiente: si hay dos espadas, el RPN genera dos RoIs separados, uno por cada espada, aunque estén tocándose. Cada RoI tiene un ID propio desde este momento.

**Etapa 2 — Clasificación + Máscara por instancia: Para cada RoI propuesto, Mask R-CNN corre en paralelo tres cabezales:** La operación RoIAlign extrae características de cada RoI de forma precisa y sin pérdida de alineación espacial, permitiendo que la máscara de cada instancia sea pixel-accurate. Entonces, ¿Por qué puede iluminar una espada de rojo y otra de azul aunque se toquen? Porque en la Etapa 1, el RPN ya las separó conceptualmente en dos RoIs independientes antes de que la Etapa 2 genere sus máscaras. Cada espada tiene su propia máscara generada en su propio contexto de RoI y la red nunca ve ambas espadas fusionadas como problema único. La espada del Usuario A tiene su propio ID con su máscara individual, la espada del Usuario B tiene otra instancia de ID con la suya. El sistema de post proceso de la cabina AR puede entonces aplicar el glow rojo al ID 1 y el azul al ID 2 de forma determinista. Incluso si los píxeles cercanos se superponen, cada máscara de instancia fue generada independientemente y puede superponerse con prioridad definida por el sistema.

## **Task 2**

**En el pabellón principal del evento, se está llevando a cabo un concurso de cosplay. De repente, un grupo de 15 personas haciendo cosplay de Naruto entran al escenario exactamente iguales y posan muy juntos (simulando el Kage Bunshin no Jutsu o Jutsu Clones de Sombra). Las cámaras registran a los clones abrazados, superpuestos y parcialmente ocluidos unos detrás de otros. Usted nota que su modelo clásico basado en YOLOv8 falla rotundamente: la red neuronal sí encuentra a los 15 clones y dibuja cientos de cajas, pero al final en la pantalla solo aparecen 4 o 5 personas detectadas. El resto "desaparece". Considerando esto, responda:**

**1. Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).**

- En este escenario, los 15 clones están parados muy juntos y sus bounding boxes se superponen de manera masiva. El IoU entre dos cajas se calcula como: 

    $$IoU(A, B) = \frac{|A \cap B|}{|A \cup B|}$$


    Cuando dos personas están casi encima una de la otra, el área de intersección es muy grande en relación al área de unión, lo que da un IoU relativamente alto. NMS interpreta eso como si las dos cajas están detectando el mismo objeto y elimina la de menor confianza. El problema es que no eran el mismo objeto, realmente eran personas distintas. Los falsos negativos se generan al descartar detecciones reales pensando que son duplicados. 

**2. Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.**

- Con un umbral de 0.15, NMS se vuelve extremadamente agresivo. Cualquier par de cajas que se solape aunque sea un poco se considera duplicado y se elimina. En un escenario como este, casi todas las cajas se descartan entre sí, y el resultado sería detectar incluso menos personas que antes.

    Con un umbral de 0.95, NMS solo elimina cajas que sean prácticamente idénticas (superpuestas en un 95%). Esto permitiría que coexistan detecciones de personas distintas aunque estén muy juntas, porque sus cajas, aunque se solapen bastante, raramente llegan a ese nivel de coincidencia.

    Para este problema en específico, el valor más adecuado sería uno alto, alrededor de 0.8-0.85. La idea es mantener las detecciones de personas cercanas y descartar únicamente las cajas que sean casi idénticas entre sí.

**3. Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de oclusión sin necesidad de tocar el NMS**

- YOLOv10 resuelve el problema con su arquitectura de Dual Label Assignment, utilizando dos cabezas de predicción en paralelo. La primera, llamada one-to-many, asigna múltiples anchors por objeto para generar una señal de entrenamiento que sea capaz de ayudar al modelo a aprender buenas representaciones. La segunda, llamada one-to-one, fuerza que cada objeto real tenga exactamente una predicción ganadora, entrenando al modelo para filtrar sus propios duplicados internamente.

    Durante la inferencia solo se usa la cabeza one-to-one. Como el modelo ya aprendió a producir una única predicción por objeto, no genera las múltiples cajas superpuestas que NMS tendría que limpiar. Esto resuelve directamente el problema de los clones, puesto que cada persona tiene su propia predicción sin que NMS tenga que intervenir y, en el proceso, descartar detecciones válidas.